# Ejercicio práctico · Costos en la nube
**Curso de Capacitación en Infraestructura Computacional para Finanzas Cuantitativas · Tema 3: Computación en la nube con aplicaciones a finanzas**

**El caso (empresa ficticia):** el equipo de riesgos de *Banco Andino* usa la nube desde hace seis meses y la factura se triplicó. Tu trabajo es entender por qué, decidir cómo conviene pagar el cómputo y proponer controles.

| Parte | Qué haces | Tiempo |
|---|---|---|
| A · Detective de la factura | Analizas seis meses de facturas con pandas | 17 min |
| B · ¿Comprar, alquilar o reservar? | Comparas cuatro formas de pagar el cómputo | 25 min |
| C · Presupuesto y alerta | Defines una alerta y una recomendación | 8 min |

**Cómo trabajar:** ejecuta las celdas en orden. En las celdas con `____` reemplaza los espacios por la respuesta. Todos los datos y precios son **ficticios**: no necesitas cuenta en ninguna nube.

## Parte A · Detective de la factura

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

meses = [1, 2, 3, 4, 5, 6]
costos = {   # costo mensual en US$ por proyecto y servicio (datos ficticios)
 ("riesgo-credito", "Cómputo"):        [600, 610, 620, 630, 640, 650],
 ("riesgo-credito", "Almacenamiento"): [150, 170, 195, 225, 260, 300],
 ("riesgo-credito", "Transferencia"):  [30, 30, 32, 30, 31, 32],
 ("riesgo-credito", "Consultas"):      [200, 210, 215, 220, 230, 240],
 ("riesgo-credito", "Otros"):          [20, 20, 20, 20, 20, 20],
 ("riesgo-mercado", "Cómputo"):        [350, 360, 360, 370, 380, 400],
 ("riesgo-mercado", "Almacenamiento"): [60, 62, 64, 66, 68, 70],
 ("riesgo-mercado", "Transferencia"):  [30, 30, 28, 30, 30, 32],
 ("riesgo-mercado", "Consultas"):      [250, 260, 270, 300, 330, 2100],
 ("marketing-analytics", "Cómputo"):        [100, 100, 105, 100, 105, 100],
 ("marketing-analytics", "Almacenamiento"): [80, 85, 90, 95, 100, 110],
 ("marketing-analytics", "Transferencia"):  [40, 45, 45, 50, 60, 1800],
 ("marketing-analytics", "Consultas"):      [150, 150, 160, 160, 170, 180],
 ("marketing-analytics", "Otros"):          [10, 10, 10, 10, 10, 10],
}
filas = [(m, p, s, v) for (p, s), vals in costos.items() for m, v in zip(meses, vals)]
df = pd.DataFrame(filas, columns=["mes", "proyecto", "servicio", "usd"])
volumen = pd.Series([900, 920, 950, 980, 1000, 1030], index=meses)   # miles de transacciones por mes
df.head()

### A1 · ¿Cuánto creció la factura?
Ya está resuelto como ejemplo: agrupa por mes y suma.

In [ ]:
total_mes = df.groupby("mes")["usd"].sum()
print(total_mes)
print("Mes 6 frente al mes 1:", round(total_mes[6] / total_mes[1], 1), "veces")

### A2 · Tabla por servicio y mes
Completa los espacios: filas = servicio, columnas = mes.

In [ ]:
tabla = df.pivot_table(index="____", columns="____", values="usd", aggfunc="sum")
tabla

### A3 · ¿Qué servicios explican el salto del mes 5 al mes 6?
Calcula el aumento de cada servicio y ordénalo de mayor a menor.

In [ ]:
aumento = (tabla[____] - tabla[____]).sort_values(ascending=False)
aumento

### A4 · ¿Y qué proyecto?
Mira solo los meses 5 y 6, por proyecto y servicio, y muestra las tres filas que más subieron.

In [ ]:
t56 = df[df["mes"].isin([5, 6])].pivot_table(index=["proyecto", "servicio"], columns="mes", values="usd", aggfunc="sum")
t56["cambio"] = t56[____] - t56[____]
t56.sort_values("cambio", ascending=False).head(3)

### A5 · La subida silenciosa
No todo es un salto: una línea puede crecer poco a poco. Calcula el crecimiento mensual del almacenamiento de `riesgo-credito`.

In [ ]:
alm = df[(df["proyecto"] == "riesgo-credito") & (df["servicio"] == "Almacenamiento")].set_index("mes")["usd"]
crecimiento = alm.____() * 100          # pista: pct_change() da el cambio porcentual
print(crecimiento.round(1))

### A6 · Gráfico
Barras apiladas por servicio y mes (ya resuelto).

In [ ]:
tabla.T.plot(kind="bar", stacked=True, figsize=(8, 4))
plt.ylabel("US$ por mes"); plt.xlabel("mes"); plt.title("Factura por servicio"); plt.show()

**Escribe tus conclusiones de la Parte A** (en esta celda, como comentarios):
- ¿Qué dos líneas explican casi todo el salto del mes 6?
- ¿Qué hipótesis tienes para cada una? (pista: descargas masivas y consultas sin filtros)
- ¿Qué línea crece sin que nadie la note?

In [ ]:
# Escribe aquí tus conclusiones:
#

## Parte B · ¿Comprar, alquilar o reservar?

La demanda de cómputo se mide en **máquina-horas por mes**. Cada trimestre hay una prueba de estrés (*stress test*: simular escenarios extremos para ver si la entidad resiste), por eso hay picos en marzo, junio, septiembre y diciembre. Un mes tiene 720 horas.

| Opción | Cómo se paga | Precio (ficticio) |
|---|---|---|
| **A · Infraestructura propia** | Máquinas para el pico, pagadas todo el año aunque no se usen | US$ 250 por máquina al mes |
| **B · Nube bajo demanda** | Solo las horas usadas | US$ 0,60 por máquina-hora |
| **C · Nube con reserva** | M máquinas reservadas las 24 horas con 40 % de descuento (US$ 0,36 por hora); el exceso se paga bajo demanda | US$ 0,36 por hora reservada |
| **D · Híbrida** | M máquinas propias para la base y la nube bajo demanda para los picos | US$ 250 por máquina propia al mes |

In [ ]:
import numpy as np, math
demanda = np.array([2000, 2100, 5200, 2000, 2100, 5400, 2200, 2000, 5600, 2100, 2200, 6000])  # máquina-horas por mes
H = 720                       # horas en un mes
P_PROPIA, P_DEMANDA, P_RESERVA = 250, 0.60, 0.36
print("Demanda total del año:", demanda.sum(), "máquina-horas | pico mensual:", demanda.max())

### B1 · Opciones A y B
**A:** necesitas máquinas para el pico: `ceil(pico / 720)`. **B:** pagas todas las horas usadas.

In [ ]:
n_pico = math.ceil(demanda.max() / ____)
costo_A = n_pico * P_PROPIA * 12
costo_B = demanda.sum() * ____
print("A · propia:", n_pico, "máquinas →", round(costo_A), "US$/año")
print("B · bajo demanda:", round(costo_B), "US$/año")

### B2 · Opciones C y D para cada M
El exceso de demanda sobre la capacidad (`M × 720`) se paga bajo demanda. Completa y mira cuál M cuesta menos.

In [ ]:
def costo_C(M):
    exceso = np.maximum(0, demanda - H * M).sum()
    return M * H * ____ * 12 + exceso * P_DEMANDA

def costo_D(M):
    exceso = np.maximum(0, demanda - H * M).sum()
    return M * P_PROPIA * 12 + exceso * P_DEMANDA

tabla_M = pd.DataFrame({"C · reserva": [costo_C(M) for M in range(10)],
                        "D · híbrida": [costo_D(M) for M in range(10)]}).round(0)
tabla_M.index.name = "M"
tabla_M

### B3 · El óptimo y la regla del punto de equilibrio
Encuentra el M que minimiza C y D. Luego calcula el **punto de equilibrio**: una máquina reservada conviene si se usa más que `precio reservado / precio bajo demanda` de las horas.

In [ ]:
M_C = tabla_M["C · reserva"].____()      # pista: idxmin() devuelve el M de menor costo
M_D = tabla_M["D · híbrida"].idxmin()
print("Mejor M en C:", M_C, "| costo:", tabla_M.loc[M_C, "C · reserva"])
print("Mejor M en D:", M_D, "| costo:", tabla_M.loc[M_D, "D · híbrida"])
print("Punto de equilibrio de la reserva:", round(P_RESERVA / ____ * 100), "% de uso")

### B4 · Comprueba la regla con la utilización de cada máquina
¿Qué porcentaje de las horas usa en promedio la máquina número 1, 2, 3…? Debería convenir reservar solo las que superan el punto de equilibrio.

In [ ]:
util = {k: np.clip((demanda - H * (k - 1)) / H, 0, 1).mean() for k in range(1, 7)}
pd.Series(util).mul(100).round(0).rename("% de uso de la máquina k")

### B5 · Resumen y gráfico

In [ ]:
resumen = pd.Series({"A · propia": costo_A, "B · bajo demanda": costo_B,
                     f"C · reserva (M={M_C})": tabla_M.loc[M_C, "C · reserva"],
                     f"D · híbrida (M={M_D})": tabla_M.loc[M_D, "D · híbrida"]}).round(0)
print(resumen)
ax = resumen.plot(kind="barh", figsize=(7, 3)); ax.invert_yaxis()
plt.xlabel("US$ por año"); plt.show()

### B6 · Sensibilidad: ¿qué pasa si la demanda crece?
Multiplica toda la demanda por un factor (por ejemplo, 1,3 = crece 30 %) y mira cómo cambia la cantidad óptima de máquinas reservadas.

In [ ]:
for f in [0.7, 1.0, 1.3, 1.6]:
    d = demanda * f
    costos_M = [M * H * P_RESERVA * 12 + np.maximum(0, d - H * M).sum() * P_DEMANDA for M in range(14)]
    print(f"Demanda x{f:.1f} → mejor M = {int(np.argmin(costos_M))}, costo = {min(costos_M):,.0f} US$/año")

**Escribe tu recomendación de la Parte B** (2 o 3 líneas): ¿qué opción elegirías y qué factores **no económicos** pesarían en la decisión? (pista: regulación, flexibilidad, personal, riesgo)

In [ ]:
# Escribe aquí tu recomendación:
#

## Parte C · Presupuesto, alerta y costo unitario

### C1 · Alerta de presupuesto
Define el presupuesto mensual como **el total del mes 1 más 20 %** y marca los meses que lo superan.

In [ ]:
presupuesto = total_mes[1] * ____
alerta = total_mes > presupuesto
print("Presupuesto mensual: US$", round(presupuesto))
print(alerta.map({True: "ALERTA", False: "ok"}))

### C2 · Costo unitario
En economía importa el **costo por unidad**: ¿cuánto cuesta procesar 1 000 transacciones de scoring cada mes?

In [ ]:
costo_unitario = total_mes / volumen
print(costo_unitario.round(2))
print("Cambio del mes 6 frente al mes 1:", round((costo_unitario[6] / costo_unitario[1] - 1) * 100), "%")

**Cierra con tus 3 recomendaciones de control** (qué haría tu equipo para que no se repita): por ejemplo, etiquetas por proyecto, presupuestos con alertas, límites de consulta, borrado automático de copias, revisión mensual de la factura.

In [ ]:
# Escribe aquí tus 3 recomendaciones:
# 1.
# 2.
# 3.